# 67 — Region-Constrained Applications

**Core statement**

The application changes. The specification chain persists.

This notebook maps region-constrained design onto concrete application domains: quantum control, photonics, microcombs, atom interferometry, machine learning, and robotics.

The goal is practical: show how a lab or engineering group can attach the same specification pipeline to its own system.

## Runtime requirements

```bash
pip install numpy pandas matplotlib scipy ipython
```

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, Ellipse, Polygon
from scipy import ndimage
from IPython.display import FileLink, display

OUT = Path("outputs/notebook_67_region_constrained_applications")
FIG = OUT / "figures"
DATA = OUT / "data"
FIG.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 11,
    "axes.titlesize": 18,
    "axes.labelsize": 12,
})

def savefig(fig, name):
    path = FIG / name
    fig.savefig(path, bbox_inches="tight")
    return path

def clean_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def box(ax, x, y, w, h, title, subtitle="", lw=2.0, rounded=True, title_size=11, subtitle_size=8):
    patch = FancyBboxPatch(
        (x,y), w,h,
        boxstyle="round,pad=0.02,rounding_size=0.025",
        fill=False, linewidth=lw, edgecolor="black"
    ) if rounded else Rectangle((x,y), w,h, fill=False, linewidth=lw, edgecolor="black")
    ax.add_patch(patch)
    ax.text(x+w/2, y+h*0.62, title, ha="center", va="center", fontsize=title_size, fontweight="bold")
    if subtitle:
        ax.text(x+w/2, y+h*0.27, subtitle, ha="center", va="center", fontsize=subtitle_size)
    return patch

def arrow(ax, start, end, lw=2.0):
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->", linewidth=lw, shrinkA=0, shrinkB=0))

def chain_horizontal(ax, items, y=0.50, x0=0.05, x1=0.95, w=None, h=0.16, title_size=10, subtitle_size=8):
    n = len(items)
    if w is None:
        w = min(0.12, (x1-x0)/(n*1.25))
    xs = np.linspace(x0, x1-w, n)
    for i,(t,s) in enumerate(items):
        box(ax, xs[i], y, w, h, t, s, title_size=title_size, subtitle_size=subtitle_size)
        if i < n-1:
            arrow(ax, (xs[i]+w+0.01, y+h/2), (xs[i+1]-0.01, y+h/2), lw=2)
    return xs

N = 280
x = np.linspace(0, 1, N)
y = np.linspace(0, 1, N)
X, Y = np.meshgrid(x, y)
threshold = 0.38

def sigmoid(z, s=0.045):
    return 1/(1+np.exp(-z/s))

def region_field(center=0.62, width=0.25, yscale=0.22, amp=1.0, bump=0.25, shift=(0,0), split=False):
    Xs = X - shift[0]
    Ys = Y - shift[1]
    plateau = sigmoid(Xs-(center-width), 0.045) * sigmoid((center+width)-Xs, 0.045)
    low_y = np.exp(-(Ys/yscale)**1.55)
    shoulder = bump*np.exp(-((Xs-(center+0.18))**2/0.018 + (Ys-0.08)**2/0.025))
    Z = amp*np.clip(plateau*low_y + shoulder, 0, 1)
    if split:
        Z -= 0.65*np.exp(-((Xs-center)**2/0.004 + (Ys-0.09)**2/0.04))
    return np.clip(Z,0,1)

def metrics(Z, threshold=threshold):
    mask = Z >= threshold
    labels, ncomp = ndimage.label(mask)
    if ncomp == 0:
        return dict(mask=mask, largest=np.zeros_like(mask), dist=np.zeros_like(Z), ncomp=0,
                    area=0.0, largest_area=0.0, max_margin=0.0, avg_margin=0.0, xstar=np.nan, ystar=np.nan)
    sizes = ndimage.sum(mask, labels, index=np.arange(1, ncomp+1))
    lid = int(np.argmax(sizes)+1)
    largest = labels == lid
    dist_px = ndimage.distance_transform_edt(largest)
    dist = dist_px / dist_px.max() if dist_px.max() else dist_px
    idx = np.unravel_index(np.argmax(dist_px), dist_px.shape)
    return dict(mask=mask, largest=largest, dist=dist, ncomp=int(ncomp),
                area=float(mask.mean()), largest_area=float(largest.mean()),
                max_margin=float(dist_px.max()/N), avg_margin=float(dist[largest].mean()) if largest.any() else 0.0,
                xstar=float(x[idx[1]]), ystar=float(y[idx[0]]))

def plot_region(ax, Z, title="", xlabel="control 1", ylabel="control 2", show_dist=False, annotate=True):
    M = metrics(Z)
    image = M["dist"] if show_dist else Z
    if show_dist:
        image = np.where(M["largest"], image, np.nan)
    im = ax.imshow(image, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
    ax.contour(X,Y,Z,levels=[threshold],colors="black",linewidths=1.8)
    ax.contour(X,Y,M["largest"].astype(float),levels=[0.5],colors="black",linewidths=2.6)
    ax.scatter([M["xstar"]],[M["ystar"]],s=80,zorder=4)
    if annotate:
        ax.text(0.05,0.88,f"|Ωc|={M['largest_area']:.2f}\nM={M['max_margin']:.2f}",transform=ax.transAxes,fontsize=9,fontweight="bold")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return im, M

## 00 Application Map

In [ ]:
applications = pd.DataFrame([
    ["quantum control", "admissible pulses", "connected pulse family", "pulse tolerance", "operating pulse"],
    ["photonics", "fabrication window", "connected device family", "fabrication margin", "device setting"],
    ["microcombs", "mode-locking region", "connected comb state", "detuning margin", "operating detuning"],
    ["atom interferometry", "stable trajectories", "common-mode path family", "phase margin", "measurement protocol"],
    ["ML systems", "feasible model class", "robust basin", "generalization margin", "trained model"],
    ["robotics", "reachable states", "safe trajectory family", "clearance margin", "controller"],
], columns=["Domain", "Ω", "Ωc", "d(∂Ωc)", "x*"])
applications.to_csv(DATA/"67_application_map.csv", index=False)

fig, ax = plt.subplots(figsize=(13,5.2))
clean_axes(ax)
ax.set_title("Region-Constrained Design Applies Across Experimental Systems")
tbl = ax.table(cellText=applications.values, colLabels=applications.columns, cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.55)
for (r,c), cell in tbl.get_celld().items():
    cell.set_linewidth(1.0)
    if r == 0:
        cell.set_text_props(fontweight="bold")
fig.text(0.5, 0.06, "The field changes; the specification chain persists.", ha="center", fontsize=13)
savefig(fig, "67_00_application_map.png")
plt.show()
applications

## 07 Domain Pipeline

In [ ]:
fig, ax = plt.subplots(figsize=(14,4.6))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Domain Pipeline")
items=[
    ("Domain physics","what can exist?"),
    ("Constraints","what fails?"),
    ("Ω","admissible set"),
    ("Ωc","largest component"),
    ("Distance field","d(∂Ωc)"),
    ("x*","robust operation"),
    ("Implementation","device / controller"),
]
chain_horizontal(ax, items, y=0.47, w=0.115, h=0.18, title_size=9, subtitle_size=7)
ax.text(0.5,0.18,"The same object changes name by field, but the specification chain remains invariant.",ha="center",fontsize=13)
savefig(fig,"67_07_domain_pipeline.png")
plt.show()

## 13 Quantum Control Example

In [ ]:
Z = region_field(center=0.58,width=0.27,yscale=0.24,bump=0.18)
fig, ax = plt.subplots(figsize=(7.5,6.2))
im,M = plot_region(ax,Z,title="Quantum Control: Robust Pulses Live Inside Ωc",xlabel="drive amplitude",ylabel="detuning / timing error",show_dist=True)
ax.text(0.13,0.78,"leakage / decoherence",fontsize=11)
ax.text(0.69,0.28,"connected\npulse family Ωc",fontsize=11,ha="center")
ax.annotate("robust pulse x*",xy=(M["xstar"],M["ystar"]),xytext=(0.47,0.18),arrowprops=dict(arrowstyle="->",linewidth=2),fontsize=11)
fig.colorbar(im, ax=ax, label="distance to failure boundary")
fig.text(0.5,0.02,"A pulse is robust because it sits inside a connected admissible control region.",ha="center",fontsize=12)
savefig(fig,"67_13_quantum_control_example.png")
plt.show()

## 17 Photonics / Microcomb Example

In [ ]:
# pump-detuning / microcomb landscape
Z = (
    0.95*np.exp(-((X-0.62)**2/0.06 + (Y-0.34)**2/0.025))
    + 0.65*np.exp(-((X-0.48)**2/0.08 + (Y-0.25)**2/0.035))
    + 0.35*np.exp(-((X-0.78)**2/0.03 + (Y-0.22)**2/0.02))
)
Z *= sigmoid(X-0.22, 0.06) * sigmoid(0.92-X, 0.06)
Z *= sigmoid(Y-0.08, 0.04) * sigmoid(0.62-Y, 0.08)
Z = np.clip(Z,0,1)

fig, ax = plt.subplots(figsize=(8,6))
im,M=plot_region(ax,Z,title="Photonics / Microcombs: Mode-Locking Region",xlabel="pump power",ylabel="detuning",show_dist=False)
ax.text(0.08,0.80,"under-driven",fontsize=11)
ax.text(0.72,0.78,"unstable",fontsize=11)
ax.text(0.52,0.34,"mode-locked Ωc",fontsize=12,ha="center")
ax.annotate("operating detuning x*",xy=(M["xstar"],M["ystar"]),xytext=(0.58,0.18),arrowprops=dict(arrowstyle="->",linewidth=2),fontsize=11)
fig.colorbar(im, ax=ax, label="admissibility score")
fig.text(0.5,0.02,"Fabrication and pump constraints specify the region before the device setting is selected.",ha="center",fontsize=12)
savefig(fig,"67_17_photonics_microcomb_example.png")
plt.show()

## 23 Atom Interferometry Example

In [ ]:
fig, ax = plt.subplots(figsize=(9,5.4))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Atom Interferometry: Preserve the Trajectory Family")
# paths
t=np.linspace(0,1,200)
upper_y=0.55+0.10*np.sin(np.pi*t)
lower_y=0.35-0.10*np.sin(np.pi*t)
ax.plot(0.15+0.70*t, upper_y, linewidth=3)
ax.plot(0.15+0.70*t, lower_y, linewidth=3)
ax.fill_between(0.15+0.70*t, lower_y, upper_y, alpha=0.12)
ax.text(0.50,0.73,"connected trajectory family Ωc",ha="center",fontsize=12,fontweight="bold")
# noise arrows
for xx in [0.28,0.45,0.62,0.78]:
    arrow(ax,(xx,0.86),(xx-0.03,0.68),lw=1.5)
ax.text(0.65,0.88,"common-mode noise",fontsize=11)
box(ax,0.34,0.13,0.32,0.11,"measurement protocol x*","chosen from robust path family",title_size=10,subtitle_size=7)
arrow(ax,(0.50,0.30),(0.50,0.24),lw=2)
ax.text(0.5,0.05,"Robust sensing preserves the connected family of trajectories, not only one phase estimate.",ha="center",fontsize=12)
savefig(fig,"67_23_atom_interferometry_example.png")
plt.show()

## 29 ML / Optimization Example

In [ ]:
fig, ax = plt.subplots(figsize=(8.5,6.2))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("ML / Optimization: Basins Are Regions")
# two basins
ell1=Ellipse((0.35,0.52),0.42,0.30,fill=False,linewidth=3)
ell2=Ellipse((0.72,0.44),0.22,0.52,fill=False,linewidth=3)
ax.add_patch(ell1); ax.add_patch(ell2)
ax.add_patch(Ellipse((0.35,0.52),0.25,0.15,fill=False,linestyle="--",linewidth=2))
ax.add_patch(Ellipse((0.72,0.44),0.12,0.26,fill=False,linestyle="--",linewidth=2))
ax.scatter([0.35,0.72],[0.52,0.44],s=90)
ax.text(0.35,0.72,"wide robust basin Ωc",ha="center",fontsize=12)
ax.text(0.75,0.74,"narrow high-score basin",ha="center",fontsize=12)
ax.text(0.35,0.27,"choose interior model x*",ha="center",fontsize=11)
ax.text(0.78,0.17,"fragile optimum\nnear boundary",ha="center",fontsize=11)
fig.text(0.5,0.05,"Optimization finds points; region-constrained learning preserves admissible basins.",ha="center",fontsize=12)
savefig(fig,"67_29_ml_optimization_example.png")
plt.show()

## 37 Robotics / Safety Example

In [ ]:
fig, ax = plt.subplots(figsize=(8.5,6.2))
ax.set_title("Robotics / Safety: Clearance Is Distance to Failure")
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_xlabel("workspace x"); ax.set_ylabel("workspace y")
# obstacles
obs1=Circle((0.43,0.52),0.13,fill=False,linewidth=3)
obs2=Rectangle((0.65,0.25),0.18,0.32,fill=False,linewidth=3)
ax.add_patch(obs1); ax.add_patch(obs2)
# path corridor
path_x=np.array([0.10,0.23,0.38,0.58,0.74,0.90])
path_y=np.array([0.18,0.30,0.26,0.62,0.70,0.82])
ax.plot(path_x,path_y,linewidth=3)
ax.scatter(path_x,path_y,s=45)
ax.text(0.30,0.82,"safe connected trajectory Ωc",fontsize=12)
ax.text(0.40,0.48,"obstacle",fontsize=10,ha="center")
ax.text(0.74,0.40,"obstacle",fontsize=10,ha="center")
ax.annotate("controller x*",xy=(0.58,0.62),xytext=(0.58,0.42),arrowprops=dict(arrowstyle="->",linewidth=2),fontsize=11)
ax.text(0.5,0.05,"Safe control is distance to failure boundary, not just path length.",ha="center",fontsize=12,transform=ax.transAxes)
savefig(fig,"67_37_robotics_safety_example.png")
plt.show()

## 43 Cross-Domain Correspondence

In [ ]:
corr = pd.DataFrame([
    ["Ω", "valid pulses", "device window", "feasible models", "reachable states"],
    ["Ωc", "pulse family", "stable devices", "robust basin", "safe paths"],
    ["d(∂Ωc)", "error margin", "fabrication margin", "generalization margin", "clearance"],
    ["x*", "pulse", "setting", "model", "controller"],
], columns=["Mathematical object","Quantum","Photonics","ML","Robotics"])
corr.to_csv(DATA/"67_cross_domain_correspondence.csv",index=False)

fig, ax = plt.subplots(figsize=(12,4.2))
clean_axes(ax)
ax.set_title("Cross-Domain Correspondence")
tbl=ax.table(cellText=corr.values,colLabels=corr.columns,cellLoc="center",loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1,1.55)
for (r,c),cell in tbl.get_celld().items():
    cell.set_linewidth(1)
    if r==0: cell.set_text_props(fontweight="bold")
savefig(fig,"67_43_cross_domain_correspondence.png")
plt.show()
corr

## 49 What Transfers Across Domains

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5.2))
for ax in axes:
    clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
axes[0].set_title("Transfers across domains")
for i,t in enumerate(["topology","distance","connectedness","margin","operating interior"]):
    box(axes[0],0.22,0.74-i*0.13,0.56,0.08,t,"",title_size=11)
axes[1].set_title("Does not transfer directly")
for i,t in enumerate(["hardware details","loss mechanisms","units","measurement apparatus","domain-specific noise"]):
    box(axes[1],0.20,0.74-i*0.13,0.60,0.08,t,"",title_size=11)
fig.suptitle("What Transfers Across Domains",fontsize=19)
fig.text(0.5,0.06,"Region-constrained design transfers structure, not laboratory particulars.",ha="center",fontsize=12)
savefig(fig,"67_49_what_transfers_across_domains.png")
plt.show()

## 59 Research Agenda

In [ ]:
agenda = pd.DataFrame([
    ["Quantum hardware", "map Ω from pulse simulations", "validate Ωc experimentally", "select maximum-margin gates"],
    ["Photonics / microcombs", "map pump-detuning regions", "preserve mode-locking topology", "choose robust operating windows"],
    ["ML / control", "compare basins, not only minima", "measure distance to failure", "rank architectures by Ωc"],
    ["Experimental science", "report admissible regions", "report distance fields", "report robustness margins"],
], columns=["Lane", "Step 1", "Step 2", "Step 3"])
agenda.to_csv(DATA/"67_research_agenda.csv",index=False)

fig, ax = plt.subplots(figsize=(13,4.6))
clean_axes(ax)
ax.set_title("Research Agenda")
tbl=ax.table(cellText=agenda.values,colLabels=agenda.columns,cellLoc="center",loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1,1.7)
for (r,c),cell in tbl.get_celld().items():
    cell.set_linewidth(1)
    if r==0: cell.set_text_props(fontweight="bold")
fig.text(0.5,0.06,"The next collaboration is a domain-specific Ω map with measured Ωc and reported robustness margin.",ha="center",fontsize=12)
savefig(fig,"67_59_research_agenda.png")
plt.show()
agenda

## 67 Final Thesis

In [ ]:
fig, ax = plt.subplots(figsize=(11,6.2))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Final Thesis")
ax.text(0.5,0.72,"The application changes.",ha="center",fontsize=26,fontweight="bold")
ax.text(0.5,0.58,"The specification chain persists.",ha="center",fontsize=26,fontweight="bold")
items=[("Ω","admissible set"),("Ωc","largest component"),("d(∂Ωc)","distance field"),("x*","operation")]
chain_horizontal(ax,items,y=0.30,x0=0.22,x1=0.78,w=0.13,h=0.13,title_size=13,subtitle_size=8)
ax.text(0.5,0.12,"Region-constrained design turns optimization into a consequence of admissible topology.",ha="center",fontsize=12)
savefig(fig,"67_67_final_thesis.png")
plt.show()

## 71 Export summary metadata

In [ ]:
summary = {
    "notebook": "67_region_constrained_applications",
    "thesis": "The application changes; the specification chain persists.",
    "chain": ["Ω", "Ωc", "d(∂Ωc)", "x*"],
    "domains": ["quantum control", "photonics", "microcombs", "atom interferometry", "ML systems", "robotics"],
    "figure_count": len(list(FIG.glob("*.png"))),
    "data_files": sorted([p.name for p in DATA.glob("*")]),
}
(DATA/"67_applications_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

## 79 Package and download outputs

In [ ]:
# ============================================================
# Package and download Notebook 67 outputs
# ============================================================
#
# Run this final cell after executing the notebook.
#
# It creates:
#   67_region_constrained_applications_outputs.zip
#
# containing:
#   figures/*.png
#   data/*.csv
#   data/*.json
#   README_67_outputs.md
#
# In Google Colab, it triggers an automatic browser download.
# In Jupyter, it displays a clickable download link.

package_root = Path("outputs/notebook_67_region_constrained_applications")
figure_dir = package_root / "figures"
data_dir = package_root / "data"

figure_files = sorted(figure_dir.glob("*.png"))
data_files = sorted([p for p in data_dir.glob("*") if p.is_file()])

manifest = package_root / "README_67_outputs.md"
manifest.write_text(
    "# Notebook 67 Outputs\n\n"
    "Generated from `67_region_constrained_applications.ipynb`.\n\n"
    "## Figures\n"
    + ("\n".join(f"- figures/{p.name}" for p in figure_files) if figure_files else "- No figures found. Run all cells before packaging.")
    + "\n\n## Data\n"
    + ("\n".join(f"- data/{p.name}" for p in data_files) if data_files else "- No data files found. Run all cells before packaging.")
    + "\n",
    encoding="utf-8",
)

archive_path = Path(shutil.make_archive("67_region_constrained_applications_outputs", "zip", root_dir=package_root)).resolve()

print(f"Packaged outputs: {archive_path}")
print(f"Figures: {len(figure_files)}")
print(f"Data files: {len(data_files)}")

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    display(FileLink(str(archive_path)))